# ML Tutor — Week 5: Feature engineering & validation
**Track:** Core ML · **Lab:** Lab 5 — Improve readmission prediction without cheating

**Objectives this week:**
1. Engineer features: encoding, scaling, binning, and interactions. *(CO5)*
2. Apply cross-validation and recognize/prevent data leakage. *(CO5)*
3. Improve a readmission-prediction model through iterative feature work. *(CO5)*

> Anti-shortcut rule: hints live in ML Tutor, revealed one tier at a time. Work here, check hints there. You are graded on unaided competence.


## Before you start (~25 min)
**Goal for this session:** Arrive ready to reuse the Week 4 classification workflow and to think carefully about what information the model is allowed to know at prediction time. This week is about making features better without accidentally cheating. Reference delta: keep the main path short and make feature-engineering variants optional so leakage discipline stays visible.

**Warm up — answer these from memory before scrolling on** (retrieval beats re-reading):
1. From Week 4, what is the difference between a hard prediction and a predicted probability?
2. What could go wrong if a column contains information that would only be known after the outcome occurred?
3. Name one reason categorical variables often need preprocessing before many ML models can use them.


In [ ]:
# SETUP — run me first (Shift+Enter)
import numpy as np
import pandas as pd

# Dataset: Synthetic inpatient discharge dataset with a binary 30-day readmission label and candidate predictors such as age_band, comorbidity_count, prior_admits_6m, length_of_stay, discharge_service, med_count, payer_type, weekend_discharge, and one intentionally suspicious leakage-style column for students to reject. No PHI.
# PLACEHOLDER: the dataset for this week arrives with the course repo under data/week-05/ .
# If a lab step expects a file, download it from the ml-tutor-labs repo's data/week-05/ folder first.

print("Setup OK — numpy", np.__version__, "· pandas", pd.__version__)


## Worked example — Encoding and scaling
**Lane A · the general idea:** Models usually need numeric inputs. One-hot encoding turns categories into indicator columns, and scaling puts numeric features onto comparable ranges when the model is sensitive to distance or coefficient magnitude.

**Lane B · the same idea in pharmacy terms:** A readmission dataset may encode discharge_service, insurance_type, and index_unit as categories. One-hot encoding lets the model use them without pretending one category is “larger” than another.

Below is a tiny, complete demo of this idea. Run it, read every line, change one thing, run it again.


In [ ]:
# WORKED EXAMPLE — models need numbers, on comparable scales
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.DataFrame({
    "insurance": ["private", "medicare", "medicare", "none", "private", "none"],
    "age":       [34, 71, 68, 45, 52, 39],
    "n_meds":    [1, 9, 7, 3, 4, 2],
})
encoded = pd.get_dummies(df, columns=["insurance"])       # words → 0/1 columns
print(encoded.head(3))
scaled = StandardScaler().fit_transform(encoded[["age", "n_meds"]])
print("age & n_meds rescaled to mean 0, sd 1:"); print(scaled.round(2))


## 1. Build a clean baseline validation loop
Create a baseline classification pipeline for readmission and evaluate it with cross-validation rather than one lucky split.

**Checkpoint:** Checkpoint 1 verifies: cross-validation is stratified, transforms live inside the pipeline, and the reported metrics come from cross_validate rather than a single holdout only.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression

target = "readmitted_30d"

# readmit_df: synthetic inpatient discharge records (no PHI).
readmit_df = pd.DataFrame({
    "age_band": ["18-40", "41-65", "65+", "41-65", "18-40", "65+", "41-65", "18-40",
                 "65+", "41-65", "18-40", "65+", "41-65", "18-40", "65+", "41-65"],
    "comorbidity_count": [0, 2, 4, 1, 0, 3, 2, 1, 5, 2, 0, 4, 1, 0, 3, 2],
    "prior_admits_6m": [0, 1, 3, 0, 0, 2, 1, 0, 4, 1, 0, 2, 0, 0, 3, 1],
    "length_of_stay": [2, 4, 7, 3, 1, 6, 4, 2, 9, 3, 2, 6, 3, 1, 7, 4],
    "discharge_service": ["medicine", "surgery", "medicine", "cardiology", "medicine",
                           "surgery", "cardiology", "medicine", "surgery", "medicine",
                           "cardiology", "surgery", "medicine", "medicine", "surgery", "cardiology"],
    "med_count": [3, 6, 10, 4, 2, 8, 5, 3, 12, 5, 3, 9, 4, 2, 8, 5],
    "payer_type": ["private", "medicare", "medicare", "private", "medicaid", "medicare",
                    "private", "medicaid", "medicare", "private", "medicaid", "medicare",
                    "private", "medicaid", "medicare", "private"],
    "weekend_discharge": [0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0],
    "post_discharge_followup_completed": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
    "readmitted_30d": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
})

X = ...
y = ...

categorical_cols = ["age_band", "discharge_service", "payer_type", "weekend_discharge"]
numeric_cols = ["comorbidity_count", "prior_admits_6m", "length_of_stay", "med_count"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
    ]
)

baseline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# TODO: run cross_validate with at least recall and roc_auc
scores = ...


**Self-check 1:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 2. Engineer one sensible feature change
Add one justified feature transformation such as binning, an interaction, or a derived rate, then compare cross-validated results with the baseline.

**Checkpoint:** Checkpoint 2 verifies: exactly one engineered feature is added, it is described in plain language, and students compare mean cross-validated metrics rather than cherry-picking a single fold.


In [ ]:
import pandas as pd

# readmit_df: synthetic inpatient discharge records (no PHI).
readmit_df = pd.DataFrame({
    "age_band": ["18-40", "41-65", "65+", "41-65", "18-40", "65+", "41-65", "18-40",
                 "65+", "41-65", "18-40", "65+", "41-65", "18-40", "65+", "41-65"],
    "comorbidity_count": [0, 2, 4, 1, 0, 3, 2, 1, 5, 2, 0, 4, 1, 0, 3, 2],
    "prior_admits_6m": [0, 1, 3, 0, 0, 2, 1, 0, 4, 1, 0, 2, 0, 0, 3, 1],
    "length_of_stay": [2, 4, 7, 3, 1, 6, 4, 2, 9, 3, 2, 6, 3, 1, 7, 4],
    "discharge_service": ["medicine", "surgery", "medicine", "cardiology", "medicine",
                           "surgery", "cardiology", "medicine", "surgery", "medicine",
                           "cardiology", "surgery", "medicine", "medicine", "surgery", "cardiology"],
    "med_count": [3, 6, 10, 4, 2, 8, 5, 3, 12, 5, 3, 9, 4, 2, 8, 5],
    "payer_type": ["private", "medicare", "medicare", "private", "medicaid", "medicare",
                    "private", "medicaid", "medicare", "private", "medicaid", "medicare",
                    "private", "medicaid", "medicare", "private"],
    "weekend_discharge": [0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0],
    "post_discharge_followup_completed": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
    "readmitted_30d": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
})

# Example idea placeholders — choose ONE and justify it.
# Option A: bucket length_of_stay into short / medium / long
# Option B: create an interaction like comorbidity_count * med_count
# Option C: transform prior_admits_6m into a simpler high-utilization flag

feature_df = readmit_df.copy()

# TODO: create your engineered feature here
feature_df["engineered_feature"] = ...

# TODO: rebuild X from feature_df and compare the engineered pipeline vs baseline
...


**Self-check 2:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 3. Catch and remove the leakage feature
Inspect candidate columns and exclude the one that should not be available at prediction time.

**Checkpoint:** Checkpoint 3 verifies: the suspicious future-information column is excluded, and the notebook contains a one- to two-sentence explanation tying leakage to prediction timing rather than to “bad correlation.”


In [ ]:
candidate_cols = [
    "age_band",
    "comorbidity_count",
    "prior_admits_6m",
    "length_of_stay",
    "discharge_service",
    "med_count",
    "payer_type",
    "weekend_discharge",
    "post_discharge_followup_completed",
]

# TODO: choose which column is leakage-risk and explain why
safe_cols = ...

# TODO: rebuild X using only safe_cols and rerun your validation
...


**Self-check 3:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 4. Hold out the final test set for one honest final look
After your feature decisions are settled with cross-validation, make one train/test split and report final test metrics with a short reflection on stability.

**Checkpoint:** Checkpoint 4 verifies: the test split happens after feature decisions are finalized, and the written reflection compares CV expectations with final test behavior instead of claiming certainty from one run.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, roc_auc_score

# readmit_df: synthetic inpatient discharge records (no PHI).
readmit_df = pd.DataFrame({
    "age_band": ["18-40", "41-65", "65+", "41-65", "18-40", "65+", "41-65", "18-40",
                 "65+", "41-65", "18-40", "65+", "41-65", "18-40", "65+", "41-65"],
    "comorbidity_count": [0, 2, 4, 1, 0, 3, 2, 1, 5, 2, 0, 4, 1, 0, 3, 2],
    "prior_admits_6m": [0, 1, 3, 0, 0, 2, 1, 0, 4, 1, 0, 2, 0, 0, 3, 1],
    "length_of_stay": [2, 4, 7, 3, 1, 6, 4, 2, 9, 3, 2, 6, 3, 1, 7, 4],
    "discharge_service": ["medicine", "surgery", "medicine", "cardiology", "medicine",
                           "surgery", "cardiology", "medicine", "surgery", "medicine",
                           "cardiology", "surgery", "medicine", "medicine", "surgery", "cardiology"],
    "med_count": [3, 6, 10, 4, 2, 8, 5, 3, 12, 5, 3, 9, 4, 2, 8, 5],
    "payer_type": ["private", "medicare", "medicare", "private", "medicaid", "medicare",
                    "private", "medicaid", "medicare", "private", "medicaid", "medicare",
                    "private", "medicaid", "medicare", "private"],
    "weekend_discharge": [0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0],
    "post_discharge_followup_completed": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
    "readmitted_30d": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
})

# Leakage-safe columns decided in Step 3 (post_discharge_followup_completed excluded).
safe_cols = ["age_band", "comorbidity_count", "prior_admits_6m", "length_of_stay",
             "discharge_service", "med_count", "payer_type", "weekend_discharge"]

# TODO: create X_final and y_final from your leakage-safe feature set
X_final = ...
y_final = ...

X_train, X_test, y_train, y_test = ...

categorical_cols = ["age_band", "discharge_service", "payer_type", "weekend_discharge"]
numeric_cols = ["comorbidity_count", "prior_admits_6m", "length_of_stay", "med_count"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
    ]
)

final_pipeline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

# TODO: fit your chosen pipeline on X_train, y_train
# TODO: compute test recall and test AUROC
...


**Self-check 4:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## Reflection — make it stick (5 min, write before you leave)
1. **Teach it:** explain your Step 4 code to a colleague in exactly 3 sentences — what goes in, what happens, what comes out. Write the sentences below.
2. **What would break this?** A classmate insists: *“Feature engineering means inventing lots of columns until the score goes up.”* Using what you just built, describe one concrete case from this week's lab where acting on that belief produces a wrong result — and how you would catch it.


In [ ]:
# Your reflection (write as comments)
# 1) Three sentences:
#
# 2) What would break this, concretely:
#



## Optional challenge — Homework: HW5 — Honest improvement plan for readmission prediction
Using the teaching readmission dataset, compare a baseline pipeline with one carefully engineered variant. Use cross-validation for model selection, reject at least one leakage-risk feature explicitly, then run one final holdout evaluation. Focus on explaining why each feature choice was allowed or disallowed.

**Deliverable:** One runnable notebook plus a 300-word reflection. The reflection must name one engineered feature you kept, one tempting feature you rejected as leakage, and whether the final test result matched your cross-validation expectations.


In [ ]:
# HW / challenge workspace — build it here

